### Key Intuition: Data-Level Resampling

Unlike cost-sensitive learning (which modifies the loss function), **resampling modifies the dataset itself** to balance class proportions:

1. **Random Under-Sampling (`RandomUnderSampler`):**
    
    - **Mechanism:** Randomly deletes rows from the majority class until it matches the minority count.
        
    - **Advantage:** Dramatically reduces training time and memory footprint on huge datasets.
        
    - **Major Risk:** Information loss—deleting thousands of valid majority records can remove critical decision boundaries.
        
2. **Random Over-Sampling (`RandomOverSampler`):**
    
    - **Mechanism:** Randomly duplicates existing minority class rows with replacement until it matches the majority count.
        
    - **Advantage:** Retains 100% of information across both classes.
        
    - **Major Risk:** Overfitting—the model memorizes exact duplicate rows, leading to tight, non-generalizable decision boundaries.

# 2. Resampling Techniques: Random Under-Sampling vs. Random Over-Sampling

This notebook covers:
1. **The Mechanics of Resampling**: Changing dataset shapes to balance class distributions.
2. **Random Under-Sampling (`RandomUnderSampler`)**: Discarding majority class samples (and the risk of information loss).
3. **Random Over-Sampling (`RandomOverSampler`)**: Duplicating minority class samples (and the risk of exact-match overfitting).
4. **The Golden Rule of Resampling**: Resampling **must only be applied to the training set (`X_train`, `y_train`)**, never the test set.
5. Evaluating both techniques against a baseline model.

In [2]:
# Install imbalanced-learn if not already available in your environment
# !pip install imbalanced-learn

import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler

# 1. Generate an imbalanced dataset (90:10 ratio)
X_raw, y_raw = make_classification(
    n_samples=1000,
    n_features=6,
    n_informative=4,
    n_redundant=1,
    weights=[0.90, 0.10],  # 90% Class 0, 10% Class 1
    random_state=42
)

feature_names = [f"Feature_{i+1}" for i in range(X_raw.shape[1])]
df = pd.DataFrame(X_raw, columns=feature_names)
df['Target'] = y_raw

print("=== 1. ORIGINAL RAW DATASET SHAPE & DISTRIBUTION ===")
print(f"Total rows: {len(df)}")
display(df['Target'].value_counts().to_frame(name='Count').assign(
    Proportion=df['Target'].value_counts(normalize=True).map('{:.1%}'.format)
))

=== 1. ORIGINAL RAW DATASET SHAPE & DISTRIBUTION ===
Total rows: 1000


,Count,Proportion
Target,,
0,896,89.6%
1,104,10.4%


In [3]:
display(df)

,Feature_1,Feature_2,Feature_3,Feature_4,Feature_5,Feature_6,Target
0,0.351385,2.182377,-0.255875,-0.388255,0.892724,-0.868436,1
1,-1.800301,0.169075,1.640729,-1.884878,2.475151,0.236966,0
2,-0.662567,-0.965904,-1.349986,-0.151772,0.034064,-1.147303,0
3,0.071325,1.530983,-0.103807,-0.220439,0.841334,1.106722,0
4,-0.251711,-1.705038,-1.373556,0.320075,-0.823443,-0.662904,0
...,...,...,...,...,...,...,...
995,-2.454531,1.299120,1.594082,-2.599400,3.870713,0.448045,0
996,0.048597,2.931792,-0.477828,1.577279,0.474628,2.595862,0
997,-0.116789,-1.890140,0.881585,0.328725,-1.067478,-1.546102,0
998,-0.927101,2.519026,-0.280239,1.784051,0.830151,2.617037,0


---
## Step 1: Train / Test Split (Crucial Data Leakage Rule)

> **Core Rule:** Never resample the test set. The test set must represent the true real-world distribution of your problem. Resampling is strictly a training trick.

In [5]:
X = df.drop(columns=['Target']).copy()
y = df['Target'].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

print(f"Original X_train shape: {X_train.shape}")
print(f"Train class breakdown:\n{y_train.value_counts()}")

Original X_train shape: (750, 6)
Train class breakdown:
Target
0    672
1     78
Name: count, dtype: int64


---
## Step 2: Random Under-Sampling (`RandomUnderSampler`)

* We match the majority class count down to the minority class count ($N_0 = N_1 = 80$).
* **Notice the row count drop**: Training data shrinks from $800$ rows down to $160$ rows.

---
## Step 2: Random Under-Sampling (`RandomUnderSampler`)

* We match the majority class count down to the minority class count ($N_0 = N_1 = 80$).
* **Notice the row count drop**: Training data shrinks from $800$ rows down to $160$ rows.

In [6]:
rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

print(f"Under-sampled X_train shape: {X_train_rus.shape}")
print(f"Under-sampled Target Counts:\n{y_train_rus.value_counts()}")

# Train model on under-sampled data
model_rus = LogisticRegression(random_state=42)
model_rus.fit(X_train_rus, y_train_rus)

# Predict on pristine, un-resampled test set
y_pred_rus = model_rus.predict(X_test)

print("\n=== CLASSIFICATION REPORT (RANDOM UNDER-SAMPLING) ===")
print(classification_report(y_test, y_pred_rus, target_names=['Majority (0)', 'Minority (1)']))


Under-sampled X_train shape: (156, 6)
Under-sampled Target Counts:
Target
0    78
1    78
Name: count, dtype: int64

=== CLASSIFICATION REPORT (RANDOM UNDER-SAMPLING) ===
              precision    recall  f1-score   support

Majority (0)       0.95      0.71      0.81       224
Minority (1)       0.21      0.65      0.31        26

    accuracy                           0.70       250
   macro avg       0.58      0.68      0.56       250
weighted avg       0.87      0.70      0.76       250



---
## Step 3: Random Over-Sampling (`RandomOverSampler`)

* We duplicate minority class rows until they match the majority class count ($N_1 = N_0 = 720$).
* **Notice the row count expansion**: Training data grows from $800$ rows up to $1440$ rows.

In [9]:
# Initialize and apply RandomOverSampler strictly on training data
ros = RandomOverSampler(random_state=42)
X_train_ros, y_train_ros = ros.fit_resample(X_train, y_train)

print(f"Over-sampled X_train shape: {X_train_ros.shape}")
print(f"Over-sampled Target Counts:\n{y_train_ros.value_counts()}")

# Train model on over-sampled data
model_ros = LogisticRegression(random_state=42)
model_ros.fit(X_train_ros, y_train_ros)

# Predict on pristine, un-resampled test set
y_pred_ros = model_ros.predict(X_test)

print("\n=== CLASSIFICATION REPORT (RANDOM OVER-SAMPLING) ===")
print(classification_report(y_test, y_pred_ros, target_names=['Majority (0)', 'Minority (1)']))

Over-sampled X_train shape: (1344, 6)
Over-sampled Target Counts:
Target
0    672
1    672
Name: count, dtype: int64

=== CLASSIFICATION REPORT (RANDOM OVER-SAMPLING) ===
              precision    recall  f1-score   support

Majority (0)       0.93      0.69      0.79       224
Minority (1)       0.18      0.58      0.27        26

    accuracy                           0.68       250
   macro avg       0.55      0.63      0.53       250
weighted avg       0.85      0.68      0.74       250



---
## Comparison & Trade-Off Summary

| Method | Training Data Size | Information Retained | Primary Risk | When to Use |
| :--- | :--- | :--- | :--- | :--- |
| **`RandomUnderSampler`** | Shrinks ($800 \to 160$) | **Low** (lost 640 majority rows) | Discarding critical boundary data | Massive datasets (e.g., millions of rows) where training speed/memory is a blocker. |
| **`RandomOverSampler`** | Expands ($800 \to 1440$) | **High** (all data kept) | Overfitting to exact duplicate minority samples | Small/Medium datasets with moderate imbalance where you cannot afford to lose any majority data. |

In [10]:
# 1. Combine features and target back for clean preview before sampling
df_train_before = X_train.copy()
df_train_before['Target'] = y_train

print("=== 1. BEFORE RESAMPLING (Original Training Set - First 5 Rows) ===")
print(f"Total Rows: {len(df_train_before)} | Class Counts: {dict(y_train.value_counts())}")
display(df_train_before.head())

# 2. Combine features and target back for clean preview after Random Under-Sampling
df_train_after_rus = pd.DataFrame(X_train_rus, columns=X_train.columns)
df_train_after_rus['Target'] = y_train_rus

print("\n=== 2. AFTER RANDOM UNDER-SAMPLING (First 5 Rows) ===")
print(f"Total Rows: {len(df_train_after_rus)} | Class Counts: {dict(pd.Series(y_train_rus).value_counts())}")
display(df_train_after_rus.head())

# 3. Combine features and target back for clean preview after Random Over-Sampling
df_train_after_ros = pd.DataFrame(X_train_ros, columns=X_train.columns)
df_train_after_ros['Target'] = y_train_ros

print("\n=== 3. AFTER RANDOM OVER-SAMPLING (First 5 Rows) ===")
print(f"Total Rows: {len(df_train_after_ros)} | Class Counts: {dict(pd.Series(y_train_ros).value_counts())}")
display(df_train_after_ros.head())

=== 1. BEFORE RESAMPLING (Original Training Set - First 5 Rows) ===
Total Rows: 750 | Class Counts: {0: np.int64(672), 1: np.int64(78)}


,Feature_1,Feature_2,Feature_3,Feature_4,Feature_5,Feature_6,Target
485,-0.252765,-0.375032,-1.839211,-3.630289,2.189204,-0.730005,0
578,-0.227207,-2.590921,-0.582307,-0.672445,-0.744963,-2.375998,0
288,-0.174313,1.559824,-1.256129,-0.537290,1.191106,0.774320,0
948,-1.334130,0.624072,-1.307604,-1.895893,2.387659,0.540425,0
152,-1.766684,0.660075,-0.419903,-0.262476,1.723736,0.982286,0



=== 2. AFTER RANDOM UNDER-SAMPLING (First 5 Rows) ===
Total Rows: 156 | Class Counts: {0: np.int64(78), 1: np.int64(78)}


,Feature_1,Feature_2,Feature_3,Feature_4,Feature_5,Feature_6,Target
723,0.170493,-0.175146,-0.717615,-3.348822,1.909238,0.516795,0
355,-0.248914,0.895829,0.984555,-2.273278,2.000831,0.515925,0
195,-2.826798,4.163258,-0.310369,-1.091418,4.517415,1.348132,0
889,-1.230289,-0.137746,0.204069,-2.932236,2.609431,0.377250,0
573,-0.399373,-2.669871,-1.564583,-0.656872,-0.607315,-1.505747,0



=== 3. AFTER RANDOM OVER-SAMPLING (First 5 Rows) ===
Total Rows: 1344 | Class Counts: {0: np.int64(672), 1: np.int64(672)}


,Feature_1,Feature_2,Feature_3,Feature_4,Feature_5,Feature_6,Target
0,-0.252765,-0.375032,-1.839211,-3.630289,2.189204,-0.730005,0
1,-0.227207,-2.590921,-0.582307,-0.672445,-0.744963,-2.375998,0
2,-0.174313,1.559824,-1.256129,-0.537290,1.191106,0.774320,0
3,-1.334130,0.624072,-1.307604,-1.895893,2.387659,0.540425,0
4,-1.766684,0.660075,-0.419903,-0.262476,1.723736,0.982286,0
